# Transformers

Pytorch documentation [here](https://www.pytorch.org/) \
Keras documentation [here](https://keras.io/)

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"  # Set before importing keras

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import keras
from keras import layers
from keras.datasets import imdb
from keras.utils import pad_sequences
from keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Load data - California Housing

In [ ]:

# ------------------------------------------------------------
# 1) Load data
# ------------------------------------------------------------
df = _

print(df.head())
print(df.columns)

# Target column
target_col = "median_house_value"

X = df.drop(columns=[target_col])
y = df[target_col]


In [ ]:
# ------------------------------------------------------------
# 2) Train / test split
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# ------------------------------------------------------------
# 3) Scale features
# ------------------------------------------------------------
scaler_X = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# Scale target to help neural network training
scaler_y = StandardScaler()

y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

# Convert to float32
X_train_scaled = X_train_scaled.astype("float32")
X_test_scaled = X_test_scaled.astype("float32")
y_train_scaled = y_train_scaled.astype("float32")
y_test_scaled = y_test_scaled.astype("float32")

num_features = X_train_scaled.shape[1]

print("Number of features:", num_features)

In [ ]:
# ------------------------------------------------------------
# 4) Custom layer: convert tabular features into embeddings
# ------------------------------------------------------------
class TabularEmbedding(layers.Layer):
    def __init__(self, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense = layers.Dense(embed_dim)

    def call(self, inputs):
        # inputs shape: (batch_size, num_features)
        # reshape to: (batch_size, num_features, 1)
        x = keras.ops.expand_dims(inputs, axis=-1)

        # output shape: (batch_size, num_features, embed_dim)
        return self.dense(x)

# ------------------------------------------------------------
# 3) Transformer Encoder Block
# ------------------------------------------------------------
class TransformerEncoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, inputs, training=False):
        # Self-attention
        attention_output = self.attention(inputs, inputs)
        attention_output = self.dropout1(attention_output, training=training)

        # First residual connection
        x = self.norm1(inputs + attention_output)

        # Feed-forward network
        ffn_output = self.ffn(x)
        ffn_output = self.dropout2(ffn_output, training=training)

        # Second residual connection
        return self.norm2(x + ffn_output)

## Create a graph model

In [ ]:
# ------------------------------------------------------------
# 6) Build Sequential Transformer model
# ------------------------------------------------------------
embed_dim = 32
num_heads = 4
ff_dim = 64
dropout_rate = 0.1

model = keras.Sequential([
    layers.Input(shape=(num_features,)),

    TabularEmbedding(embed_dim=embed_dim),

    TransformerEncoderBlock(
        embed_dim=embed_dim,
        num_heads=num_heads,
        ff_dim=ff_dim,
        dropout=dropout_rate
    ),

    layers.GlobalAveragePooling1D(),

    layers.Dense(64, activation="relu"),
    layers.Dropout(dropout_rate),
    layers.Dense(32, activation="relu"),

    # Regression output
    layers.Dense(1)
])


In [ ]:
model.summary()

## Define loss function and optimizer

In [ ]:
# ------------------------------------------------------------
# 7) Compile
# ------------------------------------------------------------
model.compile(loss="mse", optimizer=Adam(learning_rate=0.001), metrics=["mae"])


## Train model

In [ ]:
# ------------------------------------------------------------
# 8) Train
# ------------------------------------------------------------
history = model.fit( X_train_scaled, y_train_scaled,
                     epochs=50, batch_size=64,
                     validation_split=0.2, verbose=1)


## Plot results

In [ ]:
# ================================
# 🔟 Plot progress and results
# ================================

_, axes = plt.subplots(1,2, figsize=(10,5))
axes[0].plot(history.history['loss'], c='b')
axes[0].plot(history.history['val_loss'], c='r')
axes[1].plot(history.history['mae'], c='b')
axes[1].plot(history.history['val_mae'], c='r')
axes[0].set_title("Loss"), axes[1].set_title('MAE')

## Compute metrics over ```X_test``` images

In [ ]:
# ------------------------------------------------------------
# 9) Predict
# ------------------------------------------------------------
y_pred_scaled = model.predict(X_test_scaled)

# Convert predictions back to original dollar scale
y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_true = y_test.values.reshape(-1, 1)

# ------------------------------------------------------------
# 10) Evaluate in original scale
# ------------------------------------------------------------
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("\nEvaluation on original scale")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)